In [2]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

BASE_CSV = r"C:\Users\akvsk\Downloads\Khyathi\Project\gwl_manual_quarterly_cgwb_wb_1991_2020.csv"
OUT_SYN_CSV = r"C:\Users\akvsk\Downloads\Khyathi\Project\synthetic_gwl_india_quarterly_1991_2020.csv"

STATES = [
    ("West Bengal", (23.0, 87.0), 1.0),
    ("Telangana", (17.9, 79.0), 0.9),
    ("Andhra Pradesh", (15.9, 80.6), 1.1),
    ("Maharashtra", (19.0, 75.5), 0.8),
    ("Karnataka", (15.3, 76.5), 0.85),
    ("Tamil Nadu", (11.1, 78.7), 0.7),
    ("Uttar Pradesh", (27.0, 80.9), 0.75),
    ("Rajasthan", (26.9, 74.0), 0.35),
]

quarters = pd.period_range("1991Q1","2020Q4",freq="Q")
dates = quarters.to_timestamp(how="start")
q_num = np.array([p.quarter for p in quarters])

def climate_series(humid):
    # rainfall (mm/quarter): monsoon-heavy in Q3
    base_rain = np.select([q_num==1,q_num==2,q_num==3,q_num==4],[80,150,500,200]) * humid
    rain = np.clip(base_rain + rng.normal(0,30,size=len(quarters)), 5, None)

    # temperature (°C): hottest in Q2
    base_temp = np.select([q_num==1,q_num==2,q_num==3,q_num==4],[22,32,28,24])
    temp = base_temp + rng.normal(0,1.5,size=len(quarters)) + (1-humid)*2.0

    # relative humidity (%)
    rh = np.clip(45 + humid*35 + np.select([q_num==1,q_num==2,q_num==3,q_num==4],[5,-5,15,10]) + rng.normal(0,4,size=len(quarters)), 20, 95)

    # evapotranspiration proxy (mm)
    evap = np.clip(120 + (temp-25)*8 + (60-rh)*1.2 + rng.normal(0,15,size=len(quarters)), 50, None)
    return rain,temp,rh,evap

def gen_station_series(df_clim, humid, base_depth, pump_mult, pump_off, noise=1.2):
    t = np.arange(len(df_clim))
    trend = (0.015 + (1-humid)*0.02) * t
    seasonal = 1.8*np.sin(2*np.pi*(t/4.0) + rng.uniform(0, 2*np.pi))

    pumping_idx = np.clip(pump_off + pump_mult*df_clim["pumping_base"].values + rng.normal(0,0.02,len(t)), 0, 1.2)
    recharge_idx = np.clip(df_clim["recharge_base"].values + rng.normal(0,0.05,len(t)), -0.3, 1.8)

    rain = df_clim["rain_mm"].values
    temp = df_clim["temp_c"].values

    # gwl depth (m): deeper with pumping/temp, shallower with rain/recharge
    gwl = (base_depth + trend + 0.35*(temp-25) + 2.2*pumping_idx - 0.004*rain - 1.5*recharge_idx + seasonal
           + rng.normal(0, noise, len(t)))
    gwl = np.clip(gwl, 0.5, 80)
    return gwl, pumping_idx, recharge_idx

# ---- Load WB baseline ----
df = pd.read_csv(BASE_CSV)
df["date"] = pd.to_datetime(df["Data Acquisition Time"], errors="coerce", dayfirst=True)
df = df.dropna(subset=["date"]).rename(columns={"Groundwater Level Quarterly Manual (meter)":"gwl"})
base = df[["Station","District","State","Latitude","Longitude","date","gwl"]].dropna(subset=["gwl"]).copy()
base["quarter"] = base["date"].dt.to_period("Q")
wb_q = base.groupby(["Station","District","State","Latitude","Longitude","quarter"], as_index=False)["gwl"].mean()
wb_q["date"] = wb_q["quarter"].dt.start_time

# ---- Climate table ----
clim_rows=[]
for st,cent,humid in STATES:
    rain,temp,rh,evap = climate_series(humid)
    t = np.arange(len(quarters))
    t_norm = t/(len(quarters)-1)
    pumping_base = 0.3*t_norm + 0.05*np.sin(2*np.pi*t/12)
    recharge_base = np.clip((rain - 0.6*evap)/400, -0.2, 1.5)
    clim_rows.append(pd.DataFrame({
        "state": st, "date": dates,
        "rain_mm": rain, "temp_c": temp, "rh_pct": rh, "evap_mm": evap,
        "pumping_base": pumping_base, "recharge_base": recharge_base
    }))
climate = pd.concat(clim_rows, ignore_index=True)

# ---- WB stations: keep top-N for size control, preserve observed quarters ----
TOP_WB_STATIONS = 700
top_wb = wb_q["Station"].value_counts().head(TOP_WB_STATIONS).index
wb_st = wb_q[wb_q["Station"].isin(top_wb)].copy()
stations_wb = wb_st[["Station","District","Latitude","Longitude"]].drop_duplicates().reset_index(drop=True)

syn_rows=[]
for st,cent,humid in STATES:
    df_clim = climate[climate["state"]==st].reset_index(drop=True)

    if st == "West Bengal":
        for _, s in stations_wb.iterrows():
            sid = s["Station"]
            base_depth = float(np.clip(wb_st.loc[wb_st["Station"]==sid, "gwl"].median(), 1, 80))
            pump_mult = float(np.clip(rng.normal(1.0,0.12), 0.7, 1.4))
            pump_off  = float(np.clip(rng.normal(0.04,0.03), 0.0, 0.12))

            gwl, pumping_idx, recharge_idx = gen_station_series(df_clim, humid, base_depth, pump_mult, pump_off)

            # overwrite observed points
            obs_s = wb_st[wb_st["Station"]==sid][["date","gwl"]]
            idx_map = {d:i for i,d in enumerate(df_clim["date"].values)}
            for d, val in zip(obs_s["date"].values, obs_s["gwl"].values):
                if d in idx_map: gwl[idx_map[d]] = val

            out = df_clim[["date"]].copy()
            out["state"]=st; out["district"]=s["District"]; out["station_id"]=sid
            out["lat"]=s["Latitude"]; out["lon"]=s["Longitude"]
            out["gwl"]=gwl
            out["rain_mm"]=df_clim["rain_mm"].values
            out["temp_c"]=df_clim["temp_c"].values
            out["rh_pct"]=df_clim["rh_pct"].values
            out["evap_mm"]=df_clim["evap_mm"].values
            out["pumping_idx"]=pumping_idx
            out["recharge_idx"]=recharge_idx
            syn_rows.append(out)
    else:
        N = 220
        st_ids = [f"{st[:2].upper()}_{i:04d}" for i in range(1, N+1)]
        lat0,lon0 = cent
        lats = lat0 + rng.normal(0,1.2,size=N)
        lons = lon0 + rng.normal(0,1.5,size=N)

        for i in range(N):
            base_depth = float(np.clip(6 + (1-humid)*18 + rng.normal(0,2), 1, 40))
            pump_mult = float(np.clip(rng.normal(1.0,0.15), 0.6, 1.5))
            pump_off  = float(np.clip(rng.normal(0.05,0.03), 0.0, 0.15))

            gwl, pumping_idx, recharge_idx = gen_station_series(df_clim, humid, base_depth, pump_mult, pump_off)

            out = df_clim[["date"]].copy()
            out["state"]=st; out["district"]=f"{st.split()[0].upper()}_D{rng.integers(1,21):02d}"
            out["station_id"]=st_ids[i]
            out["lat"]=lats[i]; out["lon"]=lons[i]
            out["gwl"]=gwl
            out["rain_mm"]=df_clim["rain_mm"].values
            out["temp_c"]=df_clim["temp_c"].values
            out["rh_pct"]=df_clim["rh_pct"].values
            out["evap_mm"]=df_clim["evap_mm"].values
            out["pumping_idx"]=pumping_idx
            out["recharge_idx"]=recharge_idx
            syn_rows.append(out)

syn = pd.concat(syn_rows, ignore_index=True)
syn.to_csv(OUT_SYN_CSV, index=False)
print("Saved:", OUT_SYN_CSV, "rows:", len(syn), "states:", syn["state"].nunique())


Saved: C:\Users\akvsk\Downloads\Khyathi\Project\synthetic_gwl_india_quarterly_1991_2020.csv rows: 282120 states: 8
